# 06 · API de inferencia — Deforestación Perú

Levanta un **API FastAPI** dentro de Colab que corre el modelo (U-Net ResNet34, 8 bandas) desde el checkpoint en tu Drive, y expone una **URL pública**.

La interfaz web es aparte: `website/predict.html`. La abres (local o deployada), pegas ahí la URL pública que imprime este notebook, eliges un área + dos fechas, y el server busca las escenas Sentinel-2, corre la predicción y te devuelve la deforestación detectada.

**Orden:** corre las celdas de arriba a abajo. La última te da el link.

## 1 · Instalar dependencias

In [1]:
!pip install -q fastapi 'uvicorn[standard]' segmentation-models-pytorch \
    pystac-client rasterio pillow opencv-python-headless nest_asyncio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.5/208.5 kB 8.8 MB/s eta 0:00:00


## 2 · Montar Drive y apuntar al checkpoint
Ajusta `CKPT_PATH` si tu `.pt` está en otra ruta.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ['CKPT_PATH'] = '/content/drive/MyDrive/deforestacion-peru/data/unet_8band_pw_best.pt'
print('checkpoint existe:', os.path.exists(os.environ['CKPT_PATH']))

Mounted at /content/drive
checkpoint existe: True


## 3 · El API (modelo + Sentinel-2 + endpoints)
Define la app FastAPI `app`. Preproceso idéntico al training (notebook 05): redimensiona cada escena a 128×128, concatena antes(4)+después(4) → 8 canales, `clip(/10000, 0, 1)`. Sin offset BOA (los COGs de element84 ya vienen sin él, igual que en el training).

In [3]:
"""
API de inferencia — Deforestación Perú (detección de cambio Sentinel-2).

Sirve un endpoint que recibe un AOI (bbox) + dos fechas (antes / después),
busca la escena Sentinel-2 menos nubosa en cada ventana via STAC earth-search,
lee las bandas R,G,B,NIR recortadas al bbox, corre el U-Net (ResNet34, 8 bandas)
y devuelve la máscara de deforestación + imágenes RGB antes/después.

Diseñado para correr local (con el .pt en disco) o dentro de Colab (montando
Drive y exponiendo un endpoint público con cloudflared / proxy de Colab).

Preproceso = el MISMO que el training (notebook 05): cada escena se redimensiona
a 128x128, se concatenan antes(4)+después(4) -> 8 canales, clip(/10000, 0, 1).
Sin corrección de offset BOA: aunque el STAC anota offset -0.1 (baseline >= 04.00),
los COGs de element84 (sentinel-cogs en AWS) ya vienen sin ese offset en los
píxeles — verificado, una escena 2024 de bosque da red ~200 DN, igual que 2019.
El training tampoco restó offset (solo /10000), así que aquí hacemos lo mismo
para no romper la convención con la que aprendió el modelo.
"""
import os
import io
import math
import base64
from concurrent.futures import ThreadPoolExecutor

# --- GDAL / lectura de COGs públicos en S3 (sin firmar) ---------------------
os.environ.setdefault("AWS_NO_SIGN_REQUEST", "YES")
os.environ.setdefault("GDAL_DISABLE_READDIR_ON_OPEN", "EMPTY_DIR")
os.environ.setdefault("CPL_VSIL_CURL_ALLOWED_EXTENSIONS", ".tif")
os.environ.setdefault("GDAL_HTTP_MULTIRANGE", "YES")
os.environ.setdefault("GDAL_HTTP_MERGE_CONSECUTIVE_RANGES", "YES")

import numpy as np
import cv2
import rasterio
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling
from PIL import Image
from pystac_client import Client
import torch

from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import HTMLResponse
from pydantic import BaseModel

STAC_URL = "https://earth-search.aws.element84.com/v1"
COLLECTION = "sentinel-2-l2a"
BANDS = ["red", "green", "blue", "nir"]   # B04, B03, B02, B08 (todas 10 m)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --------------------------------------------------------------------------- #
# Modelo                                                                       #
# --------------------------------------------------------------------------- #
_MODEL = None


def _build_model():
    import segmentation_models_pytorch as smp
    # encoder_weights=None: no bajamos pesos de ImageNet, cargamos el checkpoint.
    return smp.Unet(encoder_name="resnet34", encoder_weights=None,
                    in_channels=8, classes=1)


def get_model():
    global _MODEL
    if _MODEL is not None:
        return _MODEL
    ckpt = os.environ.get("CKPT_PATH", "unet_8band_pw_best.pt")
    if not os.path.exists(ckpt):
        raise HTTPException(
            status_code=503,
            detail=(f"Checkpoint no encontrado en '{ckpt}'. "
                    "Define la variable de entorno CKPT_PATH con la ruta al "
                    ".pt (en Colab: el archivo en tu Drive)."),
        )
    model = _build_model()
    state = torch.load(ckpt, map_location=DEVICE)
    if isinstance(state, dict) and "model" in state:   # checkpoint de resume
        state = state["model"]
    model.load_state_dict(state)
    model.eval().to(DEVICE)
    _MODEL = model
    return _MODEL


# --------------------------------------------------------------------------- #
# Sentinel-2 / STAC                                                            #
# --------------------------------------------------------------------------- #
def search_scenes(bbox, start, end, max_cloud, limit=12):
    """Lista de escenas en la ventana, ordenadas por nubosidad ascendente."""
    cat = Client.open(STAC_URL)
    search = cat.search(
        collections=[COLLECTION], bbox=bbox, datetime=f"{start}/{end}",
        query={"eo:cloud_cover": {"lt": max_cloud}}, max_items=limit,
    )
    items = list(search.items())
    items.sort(key=lambda it: it.properties.get("eo:cloud_cover", 100))
    return items


def _read_band(args):
    href, bbox, size = args
    with rasterio.open(href) as src:
        vrt = WarpedVRT(src, crs="EPSG:4326", resampling=Resampling.bilinear)
        win = vrt.window(bbox[0], bbox[1], bbox[2], bbox[3])
        return vrt.read(1, window=win, out_shape=(size, size)).astype("float32")


def read_stack(item, bbox, size):
    """Lee R,G,B,NIR recortadas al bbox -> (size, size, 4) en reflectancia DN.

    Sin offset BOA: los COGs de element84 ya vienen en la convención DN/10000
    (la misma con la que se entrenó). Ver nota en el docstring del módulo.
    """
    hrefs = [(item.assets[b].href, bbox, size) for b in BANDS]
    with ThreadPoolExecutor(max_workers=4) as ex:
        bands = list(ex.map(_read_band, hrefs))
    return np.dstack(bands)   # (size, size, 4)


# --------------------------------------------------------------------------- #
# Inferencia + render                                                          #
# --------------------------------------------------------------------------- #
def predict_prob(before, after):
    """before/after = (H,W,4) DN -> mapa de probabilidad 128x128."""
    a = cv2.resize(before, (128, 128), interpolation=cv2.INTER_LINEAR)
    b = cv2.resize(after, (128, 128), interpolation=cv2.INTER_LINEAR)
    img = np.concatenate([a, b], axis=2)                 # (128,128,8)
    img = np.clip(img / 10000.0, 0, 1).astype("float32")
    x = torch.from_numpy(np.ascontiguousarray(np.transpose(img, (2, 0, 1))))
    x = x.unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        prob = torch.sigmoid(get_model()(x))[0, 0].cpu().numpy()
    return prob   # (128,128) en [0,1]


def _stretch(a, lo=2, hi=98):
    a = a.astype("float32")
    valid = a[a > 0]
    if valid.size == 0:
        return np.zeros_like(a)
    pl, ph = np.percentile(valid, [lo, hi])
    return np.clip((a - pl) / (ph - pl + 1e-6), 0, 1)


def rgb_uint8(stack):
    """(H,W,4) -> RGB uint8 con stretch por percentil (igual que el notebook)."""
    rgb = np.dstack([_stretch(stack[:, :, 0]),
                     _stretch(stack[:, :, 1]),
                     _stretch(stack[:, :, 2])])
    return (rgb * 255).astype("uint8")


def to_dataurl(arr, mode):
    im = Image.fromarray(arr, mode)
    buf = io.BytesIO()
    im.save(buf, "PNG")
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()


def bbox_area_ha(bbox):
    w, s, e, n = bbox
    R = 6371000.0
    lat = math.radians((s + n) / 2)
    dw = math.radians(e - w) * R * math.cos(lat)
    dh = math.radians(n - s) * R
    return abs(dw * dh) / 10000.0


def scene_info(item):
    return {
        "id": item.id,
        "datetime": item.properties.get("datetime", "")[:10],
        "cloud": round(item.properties.get("eo:cloud_cover", 0), 1),
        "baseline": str(item.properties.get("s2:processing_baseline", "")),
    }


# --------------------------------------------------------------------------- #
# FastAPI                                                                      #
# --------------------------------------------------------------------------- #
app = FastAPI(title="Deforestación Perú — API de inferencia")
app.add_middleware(CORSMiddleware, allow_origins=["*"],
                   allow_methods=["*"], allow_headers=["*"])


class PredictReq(BaseModel):
    bbox: list[float]            # [w, s, e, n] en lon/lat
    before: str                  # fecha objetivo "antes"  (YYYY-MM-DD)
    after: str                   # fecha objetivo "después" (YYYY-MM-DD)
    window_days: int = 45        # ventana +/- días alrededor de cada fecha
    max_cloud: float = 30.0      # nubosidad máx de la escena (%)
    threshold: float = 0.5       # umbral de la máscara (mejor F1 ~0.7)
    size: int = 384              # tamaño de las imágenes que se devuelven


class ScenesReq(BaseModel):
    bbox: list[float]
    start: str
    end: str
    max_cloud: float = 60.0


def _window(date, days):
    from datetime import datetime, timedelta
    d = datetime.strptime(date, "%Y-%m-%d")
    lo = (d - timedelta(days=days)).strftime("%Y-%m-%d")
    hi = (d + timedelta(days=days)).strftime("%Y-%m-%d")
    return lo, hi


@app.get("/api/health")
def health():
    ckpt = os.environ.get("CKPT_PATH", "unet_8band_pw_best.pt")
    return {"ok": True, "device": DEVICE, "ckpt_path": ckpt,
            "ckpt_exists": os.path.exists(ckpt), "model_loaded": _MODEL is not None}


@app.post("/api/scenes")
def scenes(req: ScenesReq):
    """Escenas disponibles en una ventana (para previsualizar fechas/nubes)."""
    items = search_scenes(req.bbox, req.start, req.end, req.max_cloud)
    return {"count": len(items), "scenes": [scene_info(it) for it in items]}


@app.post("/api/predict")
def predict(req: PredictReq):
    w, s, e, n = req.bbox
    if not (e > w and n > s):
        raise HTTPException(400, "bbox inválido, se espera [w, s, e, n].")
    # El modelo se entrena con parches por-polígono redimensionados a 128x128:
    # un AOI grande se downsamplea mucho y la deforestación fina queda sub-píxel.
    # Mejor AOI ~1-3 km. Tope duro ~50 km² para evitar resultados sin sentido.
    if bbox_area_ha(req.bbox) > 5000:
        raise HTTPException(400, "AOI muy grande (>50 km²). El modelo trabaja a "
                                 "128x128; usa un área de ~1-3 km para buen detalle.")

    b_lo, b_hi = _window(req.before, req.window_days)
    a_lo, a_hi = _window(req.after, req.window_days)

    before_items = search_scenes(req.bbox, b_lo, b_hi, req.max_cloud)
    after_items = search_scenes(req.bbox, a_lo, a_hi, req.max_cloud)
    if not before_items:
        raise HTTPException(404, f"Sin escena 'antes' en {b_lo}..{b_hi} "
                                 f"con nube <{req.max_cloud}%. Sube la nube o cambia fecha.")
    if not after_items:
        raise HTTPException(404, f"Sin escena 'después' en {a_lo}..{a_hi} "
                                 f"con nube <{req.max_cloud}%. Sube la nube o cambia fecha.")

    item_b, item_a = before_items[0], after_items[0]
    stack_b = read_stack(item_b, req.bbox, req.size)
    stack_a = read_stack(item_a, req.bbox, req.size)

    prob = predict_prob(stack_b, stack_a)
    mask128 = (prob > req.threshold).astype("uint8")
    mask = cv2.resize(mask128, (req.size, req.size), interpolation=cv2.INTER_NEAREST)

    # PNG RGBA de la máscara (rojo translúcido donde hay cambio)
    rgba = np.zeros((req.size, req.size, 4), "uint8")
    rgba[..., 0] = 255
    rgba[..., 3] = (mask * 190).astype("uint8")

    frac = float(mask128.mean())
    area_ha = bbox_area_ha(req.bbox)

    return {
        "bounds": req.bbox,
        "rgb_before": to_dataurl(rgb_uint8(stack_b), "RGB"),
        "rgb_after": to_dataurl(rgb_uint8(stack_a), "RGB"),
        "mask": to_dataurl(rgba, "RGBA"),
        "before_scene": scene_info(item_b),
        "after_scene": scene_info(item_a),
        "stats": {
            "threshold": req.threshold,
            "deforested_fraction": round(frac, 4),
            "deforested_ha": round(frac * area_ha, 1),
            "area_ha": round(area_ha, 1),
            "device": DEVICE,
        },
    }


@app.get("/", response_class=HTMLResponse)
def index():
    import os
    if os.path.exists("predict.html"):
        return HTMLResponse(open("predict.html", encoding="utf-8").read())
    return HTMLResponse("<h1>API viva. POST /api/predict</h1>")


## 3b · Escribir la interfaz web (`predict.html`)
El server la sirve en `/`. _Generada desde `website/predict.html`._

In [4]:
%%writefile predict.html
<!DOCTYPE html>
<html lang="es">
<head>
<meta charset="utf-8" />
<meta name="viewport" content="width=device-width, initial-scale=1" />
<title>Deforestación Perú — Detección de cambio Sentinel-2</title>
<link href="https://unpkg.com/maplibre-gl@4.7.1/dist/maplibre-gl.css" rel="stylesheet" />
<script src="https://unpkg.com/maplibre-gl@4.7.1/dist/maplibre-gl.js"></script>
<style>
  :root{
    --bg:#0d1117; --panel:#161b22; --panel2:#1c232c; --line:#2a3340;
    --txt:#e6edf3; --mut:#8b98a5; --accent:#3fb950; --accent2:#58a6ff; --danger:#f85149;
  }
  *{box-sizing:border-box}
  html,body{margin:0;height:100%;font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,sans-serif;
    background:var(--bg);color:var(--txt)}
  #app{display:grid;grid-template-columns:380px 1fr;height:100vh}
  /* ---- sidebar ---- */
  aside{background:var(--panel);border-right:1px solid var(--line);overflow-y:auto;padding:18px 16px;display:flex;flex-direction:column;gap:14px}
  h1{font-size:16px;margin:0;letter-spacing:.2px}
  h1 small{display:block;color:var(--mut);font-weight:400;font-size:11px;margin-top:3px}
  .card{background:var(--panel2);border:1px solid var(--line);border-radius:10px;padding:12px}
  .card h2{font-size:11px;text-transform:uppercase;letter-spacing:.8px;color:var(--mut);margin:0 0 10px}
  label.field{display:block;font-size:12px;color:var(--mut);margin:8px 0 4px}
  input,select{width:100%;background:var(--bg);border:1px solid var(--line);color:var(--txt);
    border-radius:7px;padding:7px 8px;font-size:13px;font-family:inherit}
  input[type=range]{padding:0;accent-color:var(--accent)}
  .row{display:flex;gap:8px}
  .row>div{flex:1}
  .btn{width:100%;border:1px solid var(--line);background:var(--bg);color:var(--txt);border-radius:8px;
    padding:9px;font-size:13px;cursor:pointer;font-family:inherit}
  .btn:hover{border-color:var(--accent2)}
  .btn.on{border-color:var(--accent);color:var(--accent)}
  .btn.primary{background:var(--accent);color:#06210e;border-color:var(--accent);font-weight:600}
  .btn.primary:disabled{opacity:.5;cursor:not-allowed}
  .rangeval{display:flex;justify-content:space-between;font-size:12px;color:var(--mut)}
  .rangeval b{color:var(--txt)}
  .aoi-info{font-size:11px;color:var(--mut);margin-top:8px;line-height:1.5}
  .aoi-info b{color:var(--accent)}
  .hint{font-size:11px;color:var(--mut);line-height:1.45}
  .err{background:rgba(248,81,73,.12);border:1px solid var(--danger);color:#ffb3ae;
    border-radius:8px;padding:9px;font-size:12px;display:none}
  /* ---- results ---- */
  .results{display:none;flex-direction:column;gap:10px}
  .thumbs{display:grid;grid-template-columns:1fr 1fr;gap:8px}
  .thumb{position:relative;border:1px solid var(--line);border-radius:8px;overflow:hidden;aspect-ratio:1}
  .thumb img{width:100%;height:100%;object-fit:cover;display:block}
  .thumb .ov{position:absolute;inset:0;width:100%;height:100%;object-fit:cover}
  .thumb span{position:absolute;left:6px;top:6px;background:rgba(0,0,0,.6);font-size:10px;
    padding:2px 6px;border-radius:5px;color:#fff}
  .thumb.wide{grid-column:1/3;aspect-ratio:2/1}
  .stat{display:flex;justify-content:space-between;font-size:12px;padding:3px 0;border-bottom:1px solid var(--line)}
  .stat:last-child{border:0}
  .stat span{color:var(--mut)} .stat b{color:var(--txt)}
  .big{color:var(--accent);font-size:15px}
  /* ---- map ---- */
  #map{position:relative}
  .map-tools{position:absolute;top:12px;left:12px;z-index:5;display:flex;gap:8px}
  .map-tools button{background:rgba(13,17,23,.9);border:1px solid var(--line);color:var(--txt);
    border-radius:8px;padding:7px 11px;font-size:12px;cursor:pointer;backdrop-filter:blur(6px)}
  .map-tools button.on{border-color:var(--accent);color:var(--accent)}
  .overlay-ctl{position:absolute;bottom:14px;left:12px;z-index:5;background:rgba(13,17,23,.92);
    border:1px solid var(--line);border-radius:10px;padding:10px 12px;font-size:12px;display:none;
    backdrop-filter:blur(6px);min-width:190px}
  .overlay-ctl label{display:flex;align-items:center;gap:7px;margin:5px 0;cursor:pointer}
  .overlay-ctl input[type=range]{width:80px}
  .spinner{position:absolute;inset:0;z-index:9;background:rgba(13,17,23,.7);display:none;
    align-items:center;justify-content:center;flex-direction:column;gap:14px;color:var(--mut);font-size:13px}
  .spinner.on{display:flex}
  .ring{width:34px;height:34px;border:3px solid var(--line);border-top-color:var(--accent);
    border-radius:50%;animation:spin .8s linear infinite}
  @keyframes spin{to{transform:rotate(360deg)}}
</style>
</head>
<body>
<div id="app">
  <aside>
    <h1>Deforestación Perú
      <small>Detección de cambio · U-Net 8 bandas · Sentinel-2</small></h1>

    <div class="err" id="err"></div>

    <div class="card">
      <h2>1 · Endpoint de la API</h2>
      <input type="text" id="api-base" placeholder="https://...trycloudflare.com">
      <div class="hint" style="margin-top:6px">Pega la URL pública que imprime el
        notebook de Colab. Vacío = mismo origen.
        <span id="api-stat"></span></div>
    </div>

    <div class="card">
      <h2>2 · Área de interés</h2>
      <label class="field">Zonas de deforestación en Perú</label>
      <select id="hotspots"><option value="">— elige una zona —</option></select>
      <div class="row" style="margin-top:8px">
        <button class="btn" id="t-draw">✏ Dibujar AOI</button>
        <button class="btn" id="t-view">🗺 Usar vista</button>
      </div>
      <div class="aoi-info" id="aoi-info">Elige una zona conocida, o dibuja un rectángulo.</div>
      <div class="hint" style="margin-top:6px">Usa un área chica (~1-3 km). El modelo
        trabaja a 128 px; un AOI grande se ve borroso y detecta peor.</div>
    </div>

    <div class="card">
      <h2>3 · Fechas</h2>
      <div class="row">
        <div>
          <label class="field">Antes</label>
          <input type="date" id="d-before" value="2019-06-15">
        </div>
        <div>
          <label class="field">Después</label>
          <input type="date" id="d-after" value="2020-08-15">
        </div>
      </div>
      <label class="field">Ventana de búsqueda: ± <b id="win-v">45</b> días</label>
      <input type="range" id="win" min="10" max="120" value="45">
    </div>

    <div class="card">
      <h2>4 · Parámetros</h2>
      <label class="field">Nubosidad máx</label>
      <div class="rangeval"><span>0%</span><b id="cloud-v">30%</b></div>
      <input type="range" id="cloud" min="0" max="100" value="30">
      <div class="hint">La nubosidad es de la escena completa (~100 km), no del AOI:
        una escena "despejada" puede tener nube sobre tu área.</div>
      <label class="field">Umbral de detección</label>
      <div class="rangeval"><span>0.1</span><b id="thr-v">0.50</b></div>
      <input type="range" id="thr" min="10" max="90" value="50">
      <div class="hint">Umbral 0.7 da el mejor F1 en test (0.58).</div>
    </div>

    <button class="btn primary" id="run" disabled>⚡ Generar predicción</button>

    <div class="results" id="results" style="display:none">
      <div class="card">
        <h2>Resultado</h2>
        <div class="thumbs">
          <div class="thumb"><span>Antes</span><img id="r-before"></div>
          <div class="thumb"><span>Después</span><img id="r-after"></div>
          <div class="thumb wide"><span>Cambio detectado</span>
            <img id="r-after2"><img class="ov" id="r-mask"></div>
        </div>
      </div>
      <div class="card" id="stats"></div>
    </div>
  </aside>

  <div id="map">
    <div class="map-tools">
      <button id="t-draw2" class="on">Pan</button>
    </div>
    <div class="overlay-ctl" id="ov-ctl">
      <label><input type="checkbox" id="ck-after" checked> Imagen "después"</label>
      <label><input type="checkbox" id="ck-mask" checked> Máscara de cambio
        <input type="range" id="op-mask" min="0" max="100" value="90"></label>
      <label><input type="checkbox" id="ck-before"> Imagen "antes"</label>
    </div>
    <div class="spinner" id="spin"><div class="ring"></div><div id="spin-txt">Buscando escenas…</div></div>
  </div>
</div>

<script>
const $ = id => document.getElementById(id);
const apiBase = () => ($("api-base").value.trim() || location.origin).replace(/\/$/, "");
$("api-base").value = localStorage.getItem("defo_api_base") || "";
$("api-base").oninput = () => localStorage.setItem("defo_api_base", $("api-base").value.trim());

// ---- mapa ----
const map = new maplibregl.Map({
  container: "map", hash: false,
  style: { version: 8,
    sources: { gsat: { type: "raster",
      tiles: ["https://mt0.google.com/vt/lyrs=s&x={x}&y={y}&z={z}",
              "https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}",
              "https://mt2.google.com/vt/lyrs=s&x={x}&y={y}&z={z}"],
      tileSize: 256, attribution: "© Google" } },
    layers: [ {id:"bg",type:"background",paint:{"background-color":"#0d1117"}},
              {id:"gsat",type:"raster",source:"gsat"} ] },
  center: [-73.5, -9.5], zoom: 5.2,
});
map.addControl(new maplibregl.NavigationControl(), "top-right");

let bbox = null;          // [w,s,e,n]
let mode = "pan";         // pan | draw
let corner1 = null;

map.on("load", () => {
  map.addSource("aoi", {type:"geojson", data:{type:"FeatureCollection",features:[]}});
  map.addLayer({id:"aoi-fill",type:"fill",source:"aoi",
    paint:{"fill-color":"#3fb950","fill-opacity":0.12}});
  map.addLayer({id:"aoi-line",type:"line",source:"aoi",
    paint:{"line-color":"#3fb950","line-width":2}});
});

function rectGeo(b){ const[w,s,e,n]=b;
  return {type:"FeatureCollection",features:[{type:"Feature",geometry:{type:"Polygon",
    coordinates:[[[w,s],[e,s],[e,n],[w,n],[w,s]]]}}]}; }

function setBbox(b){
  bbox = [Math.min(b[0],b[2]),Math.min(b[1],b[3]),Math.max(b[0],b[2]),Math.max(b[1],b[3])];
  map.getSource("aoi").setData(rectGeo(bbox));
  const km = haversineKm(bbox);
  $("aoi-info").innerHTML = `<b>AOI fijado.</b><br>${bbox.map(v=>v.toFixed(4)).join(", ")}` +
    `<br>~${km.w.toFixed(1)} × ${km.h.toFixed(1)} km`;
  $("run").disabled = false;
}
function haversineKm(b){ const[w,s,e,n]=b, R=6371, lat=(s+n)/2*Math.PI/180;
  return { w:Math.abs((e-w)*Math.PI/180)*R*Math.cos(lat), h:Math.abs((n-s)*Math.PI/180)*R }; }

function setMode(m){
  mode = m;
  $("t-draw").classList.toggle("on", m==="draw");
  $("t-draw2").textContent = m==="draw" ? "Dibujando… (2 clics)" : "Pan";
  $("t-draw2").classList.toggle("on", m==="draw");
  map.getCanvas().style.cursor = m==="draw" ? "crosshair" : "";
  corner1 = null;
}
$("t-draw").onclick = () => setMode(mode==="draw" ? "pan" : "draw");
$("t-view").onclick = () => { const b=map.getBounds();
  setBbox([b.getWest(),b.getSouth(),b.getEast(),b.getNorth()]); setMode("pan"); };

// ---- zonas de deforestación más reportadas en Perú ----
// Centros aproximados; el usuario afina con "Dibujar AOI" si hace falta.
const HOTSPOTS = [
  {n:"La Pampa", r:"Madre de Dios", c:"minería de oro ilegal", lon:-69.86, lat:-12.92},
  {n:"Huepetuhe", r:"Madre de Dios", c:"minería de oro", lon:-70.50, lat:-13.13},
  {n:"Delta-1 / Guacamayo", r:"Madre de Dios", c:"minería de oro", lon:-70.30, lat:-12.99},
  {n:"Iberia – Tahuamanu", r:"Madre de Dios", c:"palma aceitera", lon:-69.49, lat:-11.38},
  {n:"Tamshiyacu", r:"Loreto", c:"cacao (caso United Cacao)", lon:-73.16, lat:-3.98},
  {n:"Nueva Requena", r:"Ucayali", c:"palma / colonias menonitas", lon:-74.85, lat:-8.33},
  {n:"Santa Clara de Uchunya", r:"Ucayali", c:"palma / conflicto Shipibo", lon:-74.90, lat:-8.43},
  {n:"Colonia Menonita Masisea", r:"Ucayali", c:"tala (menonitas)", lon:-74.32, lat:-8.62},
  {n:"Aguaytía – Padre Abad", r:"Ucayali", c:"palma aceitera", lon:-75.50, lat:-9.04},
  {n:"Pichis – Palcazú", r:"Pasco", c:"agricultura / ganadería", lon:-75.00, lat:-10.00},
  {n:"VRAEM", r:"Ayacucho / Cusco", c:"cultivos de coca", lon:-73.90, lat:-12.60},
  {n:"Tocache", r:"San Martín", c:"palma / coca", lon:-76.51, lat:-8.18},
];
HOTSPOTS.forEach((h,i) => {
  const o = document.createElement("option");
  o.value = i; o.textContent = `${h.n} — ${h.r} (${h.c})`;
  $("hotspots").appendChild(o);
});
function boxAround(lon, lat, km){
  const half = km/2, dLat = half/111, dLon = half/(111*Math.cos(lat*Math.PI/180));
  return [lon-dLon, lat-dLat, lon+dLon, lat+dLat];
}
$("hotspots").onchange = e => {
  const h = HOTSPOTS[+e.target.value];
  if(!h) return;
  setMode("pan");
  map.flyTo({center:[h.lon,h.lat], zoom:13.2});
  setBbox(boxAround(h.lon, h.lat, 2.5));   // AOI ~2.5 km listo para generar
  $("aoi-info").innerHTML = `<b>${h.n}</b> · ${h.r}<br>${h.c}` +
    `<br>${bbox.map(v=>v.toFixed(4)).join(", ")} · ~2.5 km`;
};

map.on("click", e => {
  if (mode!=="draw") return;
  const p=[e.lngLat.lng,e.lngLat.lat];
  if(!corner1){ corner1=p; } else {
    setBbox([corner1[0],corner1[1],p[0],p[1]]); setMode("pan");
  }
});
map.on("mousemove", e => {
  if(mode!=="draw"||!corner1) return;
  const p=[e.lngLat.lng,e.lngLat.lat];
  map.getSource("aoi").setData(rectGeo([corner1[0],corner1[1],p[0],p[1]]));
});

// ---- sliders ----
$("win").oninput = e => $("win-v").textContent = e.target.value;
$("cloud").oninput = e => $("cloud-v").textContent = e.target.value+"%";
$("thr").oninput = e => $("thr-v").textContent = (e.target.value/100).toFixed(2);
$("op-mask").oninput = e => { if(map.getLayer("ov-mask"))
  map.setPaintProperty("ov-mask","raster-opacity", e.target.value/100); };

function toggleLayer(id, on){ if(map.getLayer(id))
  map.setLayoutProperty(id,"visibility", on?"visible":"none"); }
$("ck-after").onchange = e => toggleLayer("ov-after", e.target.checked);
$("ck-before").onchange = e => toggleLayer("ov-before", e.target.checked);
$("ck-mask").onchange = e => toggleLayer("ov-mask", e.target.checked);

function showErr(msg){ const el=$("err"); el.textContent=msg; el.style.display="block"; }
function clearErr(){ $("err").style.display="none"; }

function imgCoords(b){ const[w,s,e,n]=b; return [[w,n],[e,n],[e,s],[w,s]]; }
function setOverlay(id, url, b, opacity){
  const coords=imgCoords(b);
  if(map.getLayer(id)) map.removeLayer(id);
  if(map.getSource(id)) map.removeSource(id);
  map.addSource(id,{type:"image",url,coordinates:coords});
  map.addLayer({id,type:"raster",source:id,paint:{"raster-opacity":opacity,"raster-fade-duration":0}});
}

// ---- predicción ----
$("run").onclick = async () => {
  if(!bbox) return;
  clearErr();
  $("spin").classList.add("on"); $("run").disabled=true; $("spin-txt").textContent="Buscando escenas…";
  const body = {
    bbox, before:$("d-before").value, after:$("d-after").value,
    window_days:+$("win").value, max_cloud:+$("cloud").value,
    threshold:+$("thr").value/100, size:384,
  };
  try {
    const r = await fetch(apiBase()+"/api/predict", {
      method:"POST", headers:{"Content-Type":"application/json"}, body:JSON.stringify(body) });
    const data = await r.json();
    if(!r.ok) throw new Error(data.detail || ("HTTP "+r.status));

    $("r-before").src=data.rgb_before; $("r-after").src=data.rgb_after;
    $("r-after2").src=data.rgb_after; $("r-mask").src=data.mask;
    $("results").style.display="flex"; $("ov-ctl").style.display="block";

    setOverlay("ov-before", data.rgb_before, data.bounds, 1);
    setOverlay("ov-after", data.rgb_after, data.bounds, 1);
    setOverlay("ov-mask", data.mask, data.bounds, $("op-mask").value/100);
    toggleLayer("ov-before", $("ck-before").checked);
    toggleLayer("ov-after", $("ck-after").checked);
    map.fitBounds([[data.bounds[0],data.bounds[1]],[data.bounds[2],data.bounds[3]]],{padding:60});

    const st=data.stats, sb=data.before_scene, sa=data.after_scene;
    $("stats").innerHTML =
      `<h2>Estadísticas</h2>`+
      `<div class="stat"><span>Deforestación</span><b class="big">${st.deforested_ha} ha</b></div>`+
      `<div class="stat"><span>% del AOI</span><b>${(st.deforested_fraction*100).toFixed(1)}%</b></div>`+
      `<div class="stat"><span>Área AOI</span><b>${st.area_ha} ha</b></div>`+
      `<div class="stat"><span>Umbral</span><b>${st.threshold}</b></div>`+
      `<div class="stat"><span>Escena antes</span><b>${sb.datetime} · ${sb.cloud}% nube</b></div>`+
      `<div class="stat"><span>Escena después</span><b>${sa.datetime} · ${sa.cloud}% nube</b></div>`+
      `<div class="stat"><span>Cómputo</span><b>${st.device}</b></div>`;
  } catch(err){
    showErr("Error: "+err.message);
  } finally {
    $("spin").classList.remove("on"); $("run").disabled=false;
  }
};
</script>
</body>
</html>


Writing predict.html


## 4 · Arrancar el server (en un hilo)

In [5]:
import time, threading, nest_asyncio, uvicorn
nest_asyncio.apply()
config = uvicorn.Config(app, host='0.0.0.0', port=8000, log_level='warning')
server = uvicorn.Server(config)
threading.Thread(target=server.run, daemon=True).start()
time.sleep(3)

import urllib.request, json
print('health:', json.load(urllib.request.urlopen('http://localhost:8000/api/health')))

health: {'ok': True, 'device': 'cuda', 'ckpt_path': '/content/drive/MyDrive/deforestacion-peru/data/unet_8band_pw_best.pt', 'ckpt_exists': True, 'model_loaded': False}


## 5 · Exponer la URL pública
Corre **una** opción.

### Opción A — cloudflared (URL pública para compartir, sin cuenta)
Genera una URL `*.trycloudflare.com`. Pégala en `predict.html`.

In [6]:
import subprocess, re, threading

!wget -q -O /tmp/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /tmp/cloudflared

proc = subprocess.Popen(
    ['/tmp/cloudflared', 'tunnel', '--url', 'http://localhost:8000', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

public_url = None
for line in proc.stdout:
    m = re.search(r'https://[-\w]+\.trycloudflare\.com', line)
    if m:
        public_url = m.group(0); break

threading.Thread(target=lambda: [None for _ in proc.stdout], daemon=True).start()
print('URL PÚBLICA:', public_url)
print('Pega esta URL en el campo "Endpoint de la API" de predict.html')

URL PÚBLICA: https://pubs-chronicle-variables-patient.trycloudflare.com
Pega esta URL en el campo "Endpoint de la API" de predict.html


### Opción B — proxy de Colab (sin instalar nada, solo para ti)
Da un link que funciona en **tu** navegador. Útil para probar rápido, pero ese link no sirve como endpoint para una `predict.html` en otro origen.

In [7]:
# Proxy de Colab: link que abre en TU navegador (esta sesión).
# OJO: NO sirve como endpoint para una predict.html en otro origen
# (pide auth de Google). Para eso usa cloudflared (opción A de arriba).
from google.colab.output import eval_js
url = eval_js('google.colab.kernel.proxyPort(8000)')
print('URL (solo tu navegador):', url)

URL (solo tu navegador): https://8000-gpu-t4-s-kkb-usw4b1-3hlhs81i43jfk-b.us-west4-1.prod.colab.dev


## 6 · Usar la app
1. Corriste la celda de **cloudflared** (Opción A) → copió una URL `https://xxxx.trycloudflare.com`.
2. **Abre esa URL en el navegador** → ahí está la app directamente (el server sirve `predict.html`).
3. Elige una **zona de deforestación**, fecha **antes** y **después**, y clic **Generar predicción**.

No tienes que pegar ninguna URL: al abrir la app desde la misma URL de cloudflared, ya habla con su propio `/api/predict`.

Si sale *“sin escena”*: sube la nubosidad máx o amplía la ventana de días.